# Phase 5 Worksheet — Retrieval Techniques, All Built on Chroma
**Corrected in this version:** LLM calls go through `ask()` (correctly routed per model) instead of `multimodal_chat()`; embeddings go through `embedder`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))  # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=200):
    """Drop-in replacement for the old multimodal_chat() text-only calls --
    correctly routed per-model via get_chat_model(), unlike inhouse_llm.py's
    own chat()/multimodal_chat() which always hit the Qwen3-14B endpoint."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=200):
    """Drop-in replacement for multimodal_chat() WITH an image -- uses the
    corrected image_url content-block format, and an actual client for the
    vision model (inhouse_llm.py never created one)."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

import chromadb
client = chromadb.HttpClient(host="localhost", port=8000)  # adjust to your Chroma server
print("Setup OK")

In [ ]:
collection = client.get_or_create_collection("phase5_retrieval")
docs = [
    ("d1", "Payment API returns a 500 error in production under high load.", {"service": "Payment", "environment": "production", "status": "Failed"}),
    ("d2", "Auth API works correctly in production with no reported issues.", {"service": "Auth", "environment": "production", "status": "OK"}),
    ("d3", "Payment API integration tests pass in the staging environment.", {"service": "Payment", "environment": "staging", "status": "OK"}),
    ("d4", "Settlement service occasionally times out during nightly batch jobs.", {"service": "Settlement", "environment": "production", "status": "Failed"}),
]
ids = [d[0] for d in docs]
texts = [d[1] for d in docs]
metadatas = [d[2] for d in docs]
embeddings = embedder.embed_documents(texts)
collection.upsert(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
print("Indexed", collection.count(), "docs")

## 1. Similarity search (baseline)

In [ ]:
q_vec = embedder.embed_query("which APIs are failing in production?")
print(collection.query(query_embeddings=[q_vec], n_results=4)["documents"][0])

## 2. Metadata filtering

In [ ]:
result = collection.query(query_embeddings=[q_vec], n_results=4,
                           where={"$and": [{"environment": "production"}, {"status": "Failed"}]})
print(result["documents"][0])

## 3. Self-query retrieval (LLM extracts the filter automatically)

In [ ]:
import json

user_query = "Show failed payment APIs in production"
schema_prompt = """Extract metadata filters from the user's query as JSON.
Valid fields: service (string), environment (production|staging), status (OK|Failed).
Respond with ONLY JSON: {"filters": {...}, "semantic_query": "..."}"""

raw = ask(schema_prompt, user_query, model=MODEL_QWEN3_14B, max_tokens=150)
print("Raw extraction:", raw)
extracted = json.loads(raw)
print("Filters:", extracted["filters"])

where_clause = {"$and": [{k: v} for k, v in extracted["filters"].items()]} if len(extracted["filters"]) > 1 else extracted["filters"]
q_vec2 = embedder.embed_query(extracted["semantic_query"])
result = collection.query(query_embeddings=[q_vec2], n_results=4, where=where_clause)
print("Self-query result:", result["documents"][0])

## 4. Hybrid search with Reciprocal Rank Fusion (this phase's teaser, solved)

In [ ]:
from rank_bm25 import BM25Okapi  # pip install rank_bm25 --break-system-packages

tokenized = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized)

def hybrid_search(query_text, k=4, rrf_k=60):
    bm25_scores = bm25.get_scores(query_text.lower().split())
    bm25_rank = {ids[i]: rank for rank, i in enumerate(np.argsort(bm25_scores)[::-1])}

    q_vec = embedder.embed_query(query_text)
    vec_result = collection.query(query_embeddings=[q_vec], n_results=len(ids))
    vector_rank = {doc_id: rank for rank, doc_id in enumerate(vec_result["ids"][0])}

    fused = {}
    for doc_id in ids:
        rrf_score = 1/(bm25_rank[doc_id] + rrf_k) + 1/(vector_rank[doc_id] + rrf_k)
        fused[doc_id] = rrf_score
    ranked = sorted(fused.items(), key=lambda x: -x[1])[:k]
    return [(doc_id, dict(zip(ids, texts))[doc_id], score) for doc_id, score in ranked]

import numpy as np
for doc_id, text, score in hybrid_search("Payment errors in production"):
    print(f"[{score:.4f}] {doc_id}: {text}")

## 5. Re-ranking (LLM as zero-shot reranker)

In [ ]:
def llm_rerank(query, candidates, model=MODEL_QWEN3_14B):
    listing = "\n".join(f"{i}: {c}" for i, c in enumerate(candidates))
    prompt = f"Query: {query}\nCandidates:\n{listing}\n\nReturn indices ordered most to least relevant, comma-separated."
    return ask("Be precise.", prompt, model=model, max_tokens=50)

candidates = collection.query(query_embeddings=[q_vec], n_results=4)["documents"][0]
print("Original order:", candidates)
print("Reranked order (indices):", llm_rerank("production failures", candidates))

## Teaser exercise
Implement multi-query retrieval: generate 3 rephrasings of 'production issues' with `ask()`, retrieve for each, and merge with RRF (reusing the function from section 4, generalized to N ranked lists instead of exactly 2).